In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader
import numpy as np
import seisbench.data as sbd
from sklearn.metrics import precision_score, recall_score
from sklearn.metrics import roc_auc_score, average_precision_score
import matplotlib.pyplot as plt
import time

from models import ResidualBlock, CausalConv1d  

In [89]:
class Phase2MultiTaskTCN(nn.Module):

    def __init__(self):
        super().__init__()

        # Input projection (4 channels now: Z, N, E, STA/LTA)
        self.input_conv = nn.Conv1d(4, 32, kernel_size=1)

        dilations = [1,2,4,8,16,32,64,128]

        self.blocks = nn.Sequential(
            *[ResidualBlock(32,5,d) for d in dilations]
        )

        # Detection head (same idea as Phase-1)
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Linear(32,1)

        # Phase heads
        self.p_head = nn.Conv1d(32,1,kernel_size=1)
        self.s_head = nn.Conv1d(32,1,kernel_size=1)

    def forward(self,x):

        x = self.input_conv(x)
        features = self.blocks(x)          # (B,32,3000)

        # Detection branch
        det = self.pool(features).squeeze(-1)
        det = self.fc(det)

        # Phase picking branches
        p_map = torch.sigmoid(self.p_head(features)).squeeze(1)
        s_map = torch.sigmoid(self.s_head(features)).squeeze(1)

        return det, p_map, s_map

In [90]:
model = Phase2MultiTaskTCN()
print(model)

Phase2MultiTaskTCN(
  (input_conv): Conv1d(4, 32, kernel_size=(1,), stride=(1,))
  (blocks): Sequential(
    (0): ResidualBlock(
      (conv1): CausalConv1d(
        (conv): Conv1d(32, 32, kernel_size=(5,), stride=(1,))
      )
      (norm1): GroupNorm(1, 32, eps=1e-05, affine=True)
      (conv2): CausalConv1d(
        (conv): Conv1d(32, 32, kernel_size=(5,), stride=(1,))
      )
      (norm2): GroupNorm(1, 32, eps=1e-05, affine=True)
      (relu): ReLU()
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (1): ResidualBlock(
      (conv1): CausalConv1d(
        (conv): Conv1d(32, 32, kernel_size=(5,), stride=(1,), dilation=(2,))
      )
      (norm1): GroupNorm(1, 32, eps=1e-05, affine=True)
      (conv2): CausalConv1d(
        (conv): Conv1d(32, 32, kernel_size=(5,), stride=(1,), dilation=(2,))
      )
      (norm2): GroupNorm(1, 32, eps=1e-05, affine=True)
      (relu): ReLU()
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (2): ResidualBlock(
      (conv1): CausalConv

In [91]:
checkpoint = torch.load("../res_models/backbone_models/checkpoint_epoch9.pth", map_location="cpu")
state_dict = checkpoint["model_state_dict"]

In [92]:
model.input_conv.weight.data[:, :3] = state_dict["input_conv.weight"]
model.input_conv.bias.data = state_dict["input_conv.bias"]

In [93]:
blocks_state = {
    k.replace("blocks.", ""): v
    for k, v in state_dict.items() if k.startswith("blocks.")
}

model.blocks.load_state_dict(blocks_state)

<All keys matched successfully>

In [94]:
model.fc.weight.data = state_dict["fc.weight"]
model.fc.bias.data = state_dict["fc.bias"]

In [95]:
x = torch.randn(2, 4, 3000)

det, p_map, s_map = model(x)

print(det.shape)
print(p_map.shape)
print(s_map.shape)

torch.Size([2, 1])
torch.Size([2, 3000])
torch.Size([2, 3000])


In [96]:
for p in model.blocks.parameters():
    p.requires_grad = False

for p in model.input_conv.parameters():
    p.requires_grad = False

In [97]:
for name, p in model.named_parameters():
    if p.requires_grad:
        print(name)

fc.weight
fc.bias
p_head.weight
p_head.bias
s_head.weight
s_head.bias


In [98]:
det_loss_fn = nn.BCEWithLogitsLoss()
phase_loss_fn = nn.BCELoss()

In [ ]:
dataset = sbd.STEAD()
metadata = dataset.metadata

In [ ]:
# ===== FILTER VALID DATA =====

eq_mask = (
    (metadata["trace_category"] == "earthquake_local") &
    (metadata["trace_p_arrival_sample"].notna()) &
    (metadata["trace_s_arrival_sample"].notna()) &
    (metadata["trace_p_arrival_sample"] < 3000) &
    (metadata["trace_s_arrival_sample"] < 3000)
)

noise_mask = metadata["trace_category"] == "noise"

filtered_metadata = metadata[eq_mask | noise_mask].reset_index(drop=True)

print("Filtered size:", len(filtered_metadata))


# ===== 20% SUBSET (PHASE-1 STYLE) =====

subset_metadata = filtered_metadata.sample(
    frac=0.2,
    random_state=42
).reset_index(drop=True)

print("Subset size:", len(subset_metadata))
print(subset_metadata["trace_category"].value_counts())

Filtered size: 1239998
Subset size: 248000
trace_category
earthquake_local    201170
noise                46830
Name: count, dtype: int64


In [ ]:
def compute_sta_lta(waveform, sta=50, lta=500, eps=1e-8):

    z, n, e = waveform

    energy = z**2 + n**2 + e**2

    sta_vals = np.convolve(energy, np.ones(sta)/sta, mode="same")
    lta_vals = np.convolve(energy, np.ones(lta)/lta, mode="same")

    ratio = sta_vals / (lta_vals + eps)

    ratio = ratio / np.max(ratio)

    return ratio

In [ ]:
def gaussian_pick(arrival, length=3000, sigma=10):

    t = np.arange(length)

    target = np.exp(-(t-arrival)**2/(2*sigma**2))

    return target

In [ ]:
class Phase2Dataset(Dataset):

    def __init__(self, dataset, metadata):

        self.dataset = dataset
        self.metadata = metadata.reset_index(drop=True)

    def __len__(self):

        return len(self.metadata)

    def __getitem__(self, idx):

        meta = self.metadata.iloc[idx]

        waveform = self.dataset.get_waveforms([idx])[0][:,:3000]

        waveform = waveform.astype(np.float32)

        # detection label
        y_det = 1 if meta["trace_category"]=="earthquake_local" else 0

        # STA/LTA feature
        sta_lta = compute_sta_lta(waveform)

        x = np.vstack([waveform, sta_lta])

        # phase targets
        if y_det==1:

            p_arr = int(meta["trace_p_arrival_sample"])
            s_arr = int(meta["trace_s_arrival_sample"])

            y_p = gaussian_pick(p_arr)
            y_s = gaussian_pick(s_arr)

        else:

            y_p = np.zeros(3000)
            y_s = np.zeros(3000)

        return (
            torch.tensor(x, dtype=torch.float32),
            torch.tensor([y_det], dtype=torch.float32),
            torch.tensor(y_p, dtype=torch.float32),
            torch.tensor(y_s, dtype=torch.float32)
        )

In [ ]:
train_dataset = Phase2Dataset(dataset, subset_metadata)

loader = DataLoader(train_dataset,
                    batch_size=8,
                    shuffle=True,
                    num_workers=2)

In [ ]:
batch = next(iter(loader))
x, y_det, y_p, y_s = batch

print(x.shape)
print(y_det.shape)
print(y_p.shape)
print(y_s.shape)

torch.Size([8, 4, 3000])
torch.Size([8, 1])
torch.Size([8, 3000])
torch.Size([8, 3000])


In [ ]:
print(torch.isnan(x).sum())

tensor(0)


In [ ]:
# metadata = dataset.metadata

# eq_mask = (
#     (metadata["trace_category"] == "earthquake_local") &
#     (metadata["trace_p_arrival_sample"] < 3000) &
#     (metadata["trace_s_arrival_sample"] < 3000)
# )

# noise_mask = metadata["trace_category"] == "noise"

# filtered_metadata = metadata[eq_mask | noise_mask].reset_index(drop=True)

# print("Filtered dataset size:", len(filtered_metadata))

In [ ]:
# train_dataset = Phase2Dataset(dataset, filtered_metadata)

In [ ]:
# def compute_sta_lta(waveform, sta=50, lta=500, eps=1e-8):

#     z, n, e = waveform

#     energy = z**2 + n**2 + e**2

#     sta_vals = np.convolve(energy, np.ones(sta)/sta, mode="same")
#     lta_vals = np.convolve(energy, np.ones(lta)/lta, mode="same")

#     ratio = sta_vals / (lta_vals + eps)

#     ratio = ratio / (np.max(ratio) + 1e-8)

#     ratio = np.clip(ratio, 0, 1)

#     return ratio

In [ ]:
batch = next(iter(loader))

x, y_det, y_p, y_s = batch

print(x.shape)
print(y_det.shape)
print(y_p.shape)
print(y_s.shape)

torch.Size([8, 4, 3000])
torch.Size([8, 1])
torch.Size([8, 3000])
torch.Size([8, 3000])


In [ ]:
print(torch.isnan(x).sum())
print(x.min(), x.max())

tensor(0)
tensor(-653.1085) tensor(570.9048)


In [ ]:
device = torch.device("cpu")

model = model.to(device)

In [ ]:
det_loss_fn = nn.BCEWithLogitsLoss()
phase_loss_fn = nn.BCELoss()

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)

In [ ]:
num_epochs = 5   # start small for testing

for epoch in range(num_epochs):

    model.train()

    total_loss = 0

    for x, y_det, y_p, y_s in loader:

        x = x.to(device)
        y_det = y_det.to(device)
        y_p = y_p.to(device)
        y_s = y_s.to(device)

        optimizer.zero_grad()

        det_out, p_out, s_out = model(x)

        # Detection loss
        L_det = det_loss_fn(det_out, y_det)

        # Phase losses
        L_p = phase_loss_fn(p_out, y_p)
        L_s = phase_loss_fn(s_out, y_s)

        # Total loss
        loss = L_det + L_p + L_s

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1} | Loss: {total_loss/len(loader):.4f}")

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7d8b20081670>
Traceback (most recent call last):
  File "/home/namithk/miniconda3/envs/eew/lib/python3.9/site-packages/torch/utils/data/dataloader.py", line 1664, in __del__
    self._shutdown_workers()
  File "/home/namithk/miniconda3/envs/eew/lib/python3.9/site-packages/torch/utils/data/dataloader.py", line 1647, in _shutdown_workers
    if w.is_alive():
  File "/home/namithk/miniconda3/envs/eew/lib/python3.9/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
AssertionError: can only test a child process
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7d8b20081670>
Traceback (most recent call last):
  File "/home/namithk/miniconda3/envs/eew/lib/python3.9/site-packages/torch/utils/data/dataloader.py", line 1664, in __del__
    self._shutdown_workers()
  File "/home/namithk/miniconda3/envs/eew/lib/python3.9/s

AcceleratorError: CUDA error: device-side assert triggered
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


In [ ]:
print(subset_metadata["trace_category"].value_counts())model.eval()

x, y_det, y_p, y_s = next(iter(loader))

x = x.to(device)

with torch.no_grad():
    det_out, p_out, s_out = model(x)

print("Detection:", torch.sigmoid(det_out[0]))
print("P peak index:", torch.argmax(p_out[0]))
print("S peak index:", torch.argmax(s_out[0]))

In [ ]:
import matplotlib.pyplot as plt

idx = 0

plt.figure(figsize=(12,4))

plt.plot(y_p[idx].cpu(), label="P target")
plt.plot(p_out[idx].cpu(), label="P pred")

plt.plot(y_s[idx].cpu(), label="S target")
plt.plot(s_out[idx].cpu(), label="S pred")

plt.legend()
plt.title("Phase Picking")
plt.show()